# DentHub (T-104) – Data Gathering, Assessment, Manual Cleaning & Leak-Free Splitting

References: `docs/ARCHITECTURE.md §11`, `docs/PRD.md §16`, `docs/TASK.md (T-104)`.

## 1. Dataset Sources & Licenses

| # | Dataset | Origin / Path | License | Usage Decision |
|---|---|---|---|---|
| 1 | **Oral Diseases** (`salmansajid05/oral-diseases`) | Kaggle (cached via `kagglehub`) | Kaggle open research / educational terms | **Included** (clean original `caries`, `calculus`, `gingivitis` folders only) |
| 2 | **Dental Cavity Detection Dataset** (`shahjahanabdullatif/dental-cavity-detection-dataset`) | Kaggle (cached via `kagglehub`, 33808 imgs / 15288 Roboflow prefixes) | **CC BY-SA 4.0** | **Excluded after manual visual audit** (see Step 2 below) |
| 3 | **Dental Caries Dataset** (Project Brief 3rd source) + Local Curated `dentalscan` | Kaggle slug returned HTTP 403 without private auth; supplemented by local curated intraoral photos (`gigi caries`, `karang gigi`, `gigi sehat`, `gusi sehat`) | Internal Capstone `CC26-PSU285` educational set | **Included** (provides verified `healthy` teeth/gums + `caries` + `calculus`) |

## 2. Step-by-Step Manual Cleaning Log (What Was Removed / Relabelled & Why)

1. **Dropped pre-augmented & duplicate export folders in `oral-diseases`** (`6757` files removed):
   - Removed `caries augmented data set/preview` (2,382), `Mouth_Ulcer_augmented_DataSet/preview` (2,541), `Tooth_discoloration_augmented_dataser/preview` (1,834), and `Caries_Gingivitus_ToothDiscoloration_Ulcer-yolo_annotated-Dataset` (1,542).
   - *Why:* Pre-augmented images share source originals with the clean folders; keeping them leaks transformed copies across train/val/test splits.
2. **Dropped out-of-scope oral conditions in `oral-diseases`** (`3241` files removed):
   - Removed `hypodontia`, `Mouth Ulcer`, and `Tooth Discoloration`.
   - *Why:* `PRD.md §5.2` and `AGENTS.md §10` restrict DentHub MVP strictly to `caries`, `calculus`, `gingivitis`, and `healthy`.
3. **Excluded `shahjahanabdullatif/dental-cavity-detection-dataset` after manual visual inspection** (`33808` files audited):
   - Inspected sample images & YOLO label files (`-13-_jpg`, `00000_png_jpg`, `-23-_jpg`, `-1-_jpg`, `-10-_jpg`, `-100-_jpg`).
   - *Findings:* No `data.yaml`/`classes.txt` ships with the dataset; class `0`-only samples included severe cavities with Korean YouTube caption banners (`00000_png_jpg`) AND severe gingivitis (`-23-_jpg`), while class `1`-only samples included severe caries (`-100-_jpg`) AND clean teeth (`-10-_jpg`), plus 90-degree Roboflow `.rf.` rotations with black corner borders.
   - *Why removed:* Treating ambiguous numeric IDs as `healthy` or `caries` would poison labels and violate mentor guidance (`PRD §6.1` quality over volume).
4. **Relabelled & unified surviving folders to canonical 4-class taxonomy**:
   - `caries orignal data set/done`, `gigi caries` → **`caries`**
   - `Calculus`, `karang gigi` → **`calculus`**
   - `Gingivitis` → **`gingivitis`**
   - `gigi sehat`, `gusi sehat` → **`healthy`**
5. **Quality & format filtering**:
   - Corrupt/unreadable dropped: **0**
   - Tiny (< 64×64 px) or extreme aspect ratio (> 4:1) dropped: **18**
6. **Cross-dataset perceptual-hash deduplication (64-bit dHash, Hamming ≤ 4)**:
   - Candidates before dedupe: **4233**
   - Exact + near-duplicates removed: **2704**
   - Unique clean images surviving: **1302**

## 3. Stratified Split BEFORE Augmentation & Anti-Leakage Proof

All clean unique images were partitioned using `StratifiedGroupKFold` grouped by perceptual-hash `cluster_id` (~70% train / ~15% val / ~15% test) **before any image augmentation** (augmentation occurs exclusively inside the training pipeline in `T-106`).

| Class | Train | Val | Test | Total |
|---|---:|---:|---:|---:|
| `caries` | 133 | 29 | 29 | 191 |
| `calculus` | 344 | 74 | 74 | 492 |
| `gingivitis` | 350 | 75 | 75 | 500 |
| `healthy` | 83 | 18 | 18 | 119 |
| **Total** | **910** | **196** | **196** | **1302** |

- **Exact dHash overlap across splits:** `0`
- **Cluster ID overlap across splits:** `0`
- **Minimum cross-split pairwise Hamming distance:** `5` (strictly > `4` — **PASS**)

In [ ]:
import json
from pathlib import Path

import pandas as pd

from wrangle import assert_no_split_leakage

PROCESSED = Path('../data/processed')
manifest = pd.read_csv(PROCESSED / 'manifest.csv')
report = json.loads((PROCESSED / 'assessment_report.json').read_text(encoding='utf-8'))
print('Class counts per split:')
print(pd.crosstab(manifest['label'], manifest['split']))
leak_check = assert_no_split_leakage(manifest, max_hamming=4)
print('Anti-leakage verification:', leak_check)